# AKOrN Training on MNIST - Google Colab

**Configuración**: 200 epochs, GPU T4, ~18 horas

**Accuracy esperada**: ~99%

---

## 📝 Instrucciones

1. Runtime → Change runtime type → **GPU (T4)**
2. Ejecuta las celdas **en orden** (1 → 2 → 3 → 4 → 5)
3. **NO re-ejecutes** la celda 2 sin hacer Runtime → Restart primero

---

## 1. Verificar GPU

In [ ]:
import torch

print("🔍 Verificando GPU...")
print(f"CUDA disponible: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.backends.cudnn.benchmark = True
    print("✅ GPU configurada")
else:
    print("⚠️ GPU no disponible - entrenamiento será muy lento")

## 2. Setup - Clonar repo e instalar dependencias

**⚠️ IMPORTANTE**: NO ejecutes esta celda más de una vez sin hacer Runtime → Restart

In [ ]:
import os
import shutil
from pathlib import Path
import subprocess

print("=" * 70)
print("SETUP: Clonar repositorio e instalar dependencias")
print("=" * 70)

# PASO 1: Limpiar y preparar
os.chdir('/content')
repo_path = Path('/content/Proyecto-Inv.-teorica.')

if repo_path.exists():
    print("\n🧹 Limpiando clone anterior...")
    shutil.rmtree(repo_path, ignore_errors=True)

# PASO 2: Clonar repositorio
print("\n📦 Clonando repositorio...")
result = subprocess.run(
    ['git', 'clone', 'https://github.com/ACRCris/Proyecto-Inv.-teorica..git'],
    capture_output=True, text=True, cwd='/content'
)
if result.returncode != 0:
    print(f"❌ Error al clonar: {result.stderr}")
    raise SystemExit("Clone falló")

print("✅ Repositorio clonado")

# PASO 3: Checkout rama mac
akorn_path = repo_path / 'codigo' / 'akorn'
print(f"\n🔄 Cambiando a rama 'mac'...")

result = subprocess.run(
    ['git', 'checkout', 'mac'],
    capture_output=True, text=True, cwd=str(akorn_path)
)
print("✅ Rama 'mac' activada")

# PASO 4: Verificar archivos críticos
print("\n📁 Verificando archivos críticos:")
archivos = [
    'train_classification.py',
    'source/__init__.py',
    'source/data/__init__.py',
    'source/data/augs.py'
]

for archivo in archivos:
    file_path = akorn_path / archivo
    if file_path.exists():
        size = file_path.stat().st_size / 1024
        print(f"  ✅ {archivo} ({size:.1f} KB)")
    else:
        print(f"  ❌ {archivo} FALTA")

# PASO 5: Instalar PyTorch
print("\n📦 Instalando PyTorch con CUDA...")
result = subprocess.run(
    ['pip', 'install', '-q', 'torch', 'torchvision', 'torchaudio', 
     '--index-url', 'https://download.pytorch.org/whl/cu118'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✅ PyTorch instalado")
else:
    print(f"⚠️ Advertencia: {result.stderr[:200]}")

# PASO 6: Instalar otras dependencias
print("\n📦 Instalando otras dependencias...")
result = subprocess.run(
    ['pip', 'install', '-q', 'tensorboard', 'tqdm', 'einops', 'ema-pytorch'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✅ Dependencias básicas instaladas")
else:
    print(f"⚠️ Advertencia: {result.stderr[:200]}")

# PASO 7: Instalar AutoAttack desde GitHub
print("\n📦 Instalando AutoAttack...")
result = subprocess.run(
    ['pip', 'install', '-q', 'git+https://github.com/fra31/auto-attack'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✅ AutoAttack instalado")
else:
    print(f"⚠️ Advertencia: {result.stderr[:200]}")

# PASO 8: Cambiar al directorio de trabajo
os.chdir(akorn_path)

print("\n" + "=" * 70)
print("✅ SETUP COMPLETADO")
print(f"📂 Directorio: {akorn_path}")
print("=" * 70)
print("\n▶️  Siguiente: Ejecuta la celda 3 (Descargar MNIST)")

## 3. Descargar MNIST

In [ ]:
import os
import torchvision
from torchvision import transforms
from pathlib import Path

# Asegurar que estamos en el directorio correcto
akorn_dir = Path('/content/Proyecto-Inv.-teorica./codigo/akorn')
os.chdir(akorn_dir)

# Crear directorio de datos
data_dir = akorn_dir / 'data'
data_dir.mkdir(exist_ok=True)

print("📥 Descargando MNIST...")
transform = transforms.ToTensor()

trainset = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
testset = torchvision.datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

print(f"✅ MNIST descargado")
print(f"   Train: {len(trainset):,} imágenes")
print(f"   Test: {len(testset):,} imágenes")

## 4. (Opcional) Montar Google Drive para guardar checkpoints

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

RESULTS_DIR = '/content/drive/MyDrive/akorn_mnist_results'
!mkdir -p {RESULTS_DIR}

print(f"✅ Checkpoints se guardarán en: {RESULTS_DIR}")

## 5. Iniciar Entrenamiento

**⏱️ Tiempo estimado**: ~18 horas en GPU T4

**Parámetros**:
- Epochs: 200
- Batch size: 128
- Capas Kuramoto: L=3, T=5, n=4
- Canales: 128

In [3]:
import os
from pathlib import Path

# Asegurar directorio correcto

print("🚀 Iniciando entrenamiento...")
print("Configuración: 200 epochs, batch=128, L=3, T=5, ch=128")
print("Tiempo estimado: ~18 horas")
print()

!python train_classification.py mnist_maxima_precision \
    --data mnist \
    --epochs 200 \
    --n 4 \
    --L 3 \
    --T 5 \
    --ch 128 \
    --batchsize 128 \
    --lr 0.0001 \
    --device cuda \
    --checkpoint_every 25 \
    --adveval_freq 10 \
    --pgdeval_freq 50

🚀 Iniciando entrenamiento...
Configuración: 200 epochs, batch=128, L=3, T=5, ch=128
Tiempo estimado: ~18 horas

/Users/acrcr/Documents/Unal-2025-2/IntroInvTeorica/proyecto/codigo/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/acrcr/Documents/Unal-2025-2/IntroInvTeorica/proyecto/codigo/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
Traceback (most recent call last):
  File "/Users/acrcr/Documents/Unal-2025-2/IntroInvTeorica/proyecto/codigo/akorn/train_classification.py", line 221, in <module>
    device = _resolve_device(args.device)
  File "/Users/acrcr/Documents/Unal-2025-2/IntroIn

## 6. Monitoreo - TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/mnist_maxima_precision

## 7. Monitoreo - Uso de GPU

In [ ]:
!nvidia-smi

## 8. Copiar checkpoints a Drive (ejecutar periódicamente)

In [ ]:
import shutil
import glob
from pathlib import Path

# Copiar checkpoints
checkpoints = glob.glob('runs/mnist_maxima_precision/checkpoint_*.pth')
ema_files = glob.glob('runs/mnist_maxima_precision/ema_*.pth')

for ckpt in checkpoints + ema_files:
    dest = f"{RESULTS_DIR}/{Path(ckpt).name}"
    shutil.copy(ckpt, dest)
    print(f"✅ Copiado: {Path(ckpt).name}")

print(f"\n✅ Total: {len(checkpoints + ema_files)} archivos copiados a Drive")

## 9. Descargar resultados (al finalizar)

In [ ]:
!zip -r mnist_results.zip runs/mnist_maxima_precision/

from google.colab import files
files.download('mnist_results.zip')

print("✅ Resultados listos para descargar")